In [1]:
import math
import pandas as pd
from pathlib import Path
import geopandas as gpd
from shapely.geometry import Point
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

# US Census Tracts for Illinois (state FIPS code 17) from the 2020 TIGER/Line shapefiles
tracts = gpd.read_file('data/tl_2020_17_tract/tl_2020_17_tract.shp')
blocks = gpd.read_file('data/tl_2020_17_tabblock10/tl_2020_17_tabblock10.shp')
city_boundary = gpd.read_file('data/city-shapefile/geo_export_41a1e3c1-d168-4dc6-9f14-0e861c9ee606.shp')

rail_stations = pd.read_csv('data/CTA_RailStations_Data-STATIC.csv')
park_facilities = pd.read_csv("data/Park_Facilities_Data-STATIC.csv")

jobs_by_census_tract_2010 = pd.read_csv('data/jobs_by_census_tract_2010.csv')
Crime_Data_2006_2010 = pd.read_csv('data/Crime_Data_2006_2010.csv')
Building_Permits_2006_2010 = pd.read_csv('data/Building_Permits_2006_2010.csv')

median_income = 48000
median_rent = 950
distance_threshold = 0.043  # approximately 5 km in degrees


def coordinates_to_census_tract(lat, lon):
    """Given a latitude and longitude, return the corresponding Census tract GEOID."""
    point = Point(lon, lat)  # Note: Point takes (x, y) which is (lon, lat)
    target_tract = tracts[tracts.contains(point)]
    return target_tract['GEOID'].iloc[0] if not target_tract.empty else pd.NA

def census_tract_to_coordinates(geoid):
    """Given a Census tract or block GEOID, return its coordinates."""
    if (len(str(geoid)) == 11):
        tract = tracts[tracts['GEOID'] == str(geoid)]
    elif (len(str(geoid)) == 15):
        tract = blocks[blocks['GEOID10'] == str(geoid)]
    if tract.empty:
        return pd.NA, pd.NA
    lat = tract.geometry.iloc[0].centroid.y
    lon = tract.geometry.iloc[0].centroid.x
    return lat, lon

def add_tract_geoid(df):
    """Create the 11-digit Census tract GEOID from state/county/tract columns."""
    df = df.copy()
    df["state"] = df["state"].astype(str).str.zfill(2)
    df["county"] = df["county"].astype(str).str.zfill(3)
    df["tract"] = df["tract"].astype(str).str.zfill(6)
    df["GEOID"] = df["state"] + df["county"] + df["tract"]
    return df

def census_tract_overlaps_city(tract_ID, city_boundary):
    """Given a Census tract and a city boundary, return whether the tract overlaps with the city."""
    tract = tracts[tracts['GEOID'] == str(tract_ID)]
    if tract.empty:
        return False
    else:
        return tract.geometry.iloc[0].within(city_boundary.geometry.iloc[0])
    
def calculate_gravity_score(df, year_var=None, min_year=None, distance=distance_threshold):
    if year_var is not None:
        return df[
        (df['distance'] < distance_threshold) & 
        (df[year_var].astype(int) >= min_year)].apply(
            lambda x: (1 / (x['distance']**2 + 1e-6)), axis=1).sum()
    else:
        return df[
            df['distance'] < distance].apply(
                lambda x: (1 / (x['distance']**2 + 1e-6)), axis=1).sum()

Forming one file that contains comparable information between two different timeframes. In this example, we are comparing between 2010 and 2020 Census Tract demographics, education, income, and wealth data.
Because Census Tracts are of different shapes between census years, a relationship file is used to convert 2010 Census Tract shapes to 2020 Census Tract shapes for fair data comparison.

In [ ]:

# Load the filtered datasets. Keep geographic IDs as strings so leading zeros survive.
data2010 = pd.read_csv("data/unprocessed_data/censusdata2010.csv",
    dtype={"state": str, "county": str, "tract": str},
)
data2020 = pd.read_csv("data/unprocessed_data/censusdata2020.csv",
    dtype={"state": str, "county": str, "tract": str},
)

# replace all invalid numeric values (-666666666) with NaN
data2010 = data2010.replace(-666666666, float('nan'), inplace=False)
data2020 = data2020.replace(-666666666, float('nan'), inplace=False)

data2010 = add_tract_geoid(data2010)
data2020 = add_tract_geoid(data2020)

# The tract relationship file maps overlapping 2010 and 2020 tract pieces.
# To compare on 2020 tract boundaries, allocate 2010 values to 2020 GEOIDs by
# the share of each 2010 tract's land area that falls inside each 2020 tract.
relationship = pd.read_csv(
    "data/unprocessed_data/tab20_tract20_tract10_st17.csv",
    dtype={"GEOID_TRACT_20": str, "GEOID_TRACT_10": str},
    usecols=[
        "GEOID_TRACT_20",
        "GEOID_TRACT_10",
        "AREALAND_TRACT_10",
        "AREALAND_PART",
    ],
)

# Keep only counties present in your local census files instead of processing
# the entire national crosswalk.
county_prefixes = set(data2010["GEOID"].str[:5]) | set(data2020["GEOID"].str[:5])
relationship = relationship[
    relationship["GEOID_TRACT_10"].str[:5].isin(county_prefixes)
    | relationship["GEOID_TRACT_20"].str[:5].isin(county_prefixes)
].copy()
relationship["weight_2010_to_2020"] = (
    relationship["AREALAND_PART"] / relationship["AREALAND_TRACT_10"]
)

# Count fields can be areal-weighted and summed from 2010 tracts into 2020
# tracts. Median fields are not additive, so they are only rough weighted
# approximations here.
count_fields = [
    'B03002_001E','B03002_002E','B03002_003E','B03002_004E','B03002_005E',
    'B03002_006E','B03002_007E','B03002_008E','B03002_009E','B03002_010E',
    'B03002_011E','B03002_012E','B15002_011E','B15002_014E','B15002_015E',
    'B15002_016E','B15002_017E','B15002_018E','B15002_028E','B15002_031E',
    'B15002_032E','B15002_033E','B15002_034E','B15002_035E','B17021_001E',
    'B25003_002E','B25003_003E',
]
median_fields = ['B19013_001E', 'B25064_001E', 'B25077_001E']
fields_to_compare = count_fields + median_fields

crosswalked2010 = relationship.merge(
    data2010[["GEOID", "NAME", *fields_to_compare]],
    left_on="GEOID_TRACT_10",
    right_on="GEOID",
    how="inner",
)

for field in fields_to_compare:
    crosswalked2010[field] = (
        crosswalked2010[field] * crosswalked2010["weight_2010_to_2020"]
    )

data2010_on_2020_boundaries = (
    crosswalked2010
    .groupby("GEOID_TRACT_20", as_index=False)[fields_to_compare]
    .sum(min_count=1)
    .rename(columns={"GEOID_TRACT_20": "GEOID"})
)

merged_data = pd.merge(
    data2010_on_2020_boundaries,
    data2020[["GEOID", "NAME", *fields_to_compare]],
    on="GEOID",
    how="inner",
    suffixes=("_2010_on_2020", "_2020"),
)

for field in fields_to_compare:
    merged_data[f'{field}_diff'] = (
        merged_data[f'{field}_2020'] - merged_data[f'{field}_2010_on_2020']
    )

# Combining all the education-related fields into a single "education_diff" field by summing the differences
merged_data['bachelors_or_higher_diff'] = (
    merged_data['B15002_015E_diff'] +
    merged_data['B15002_016E_diff'] +
    merged_data['B15002_017E_diff'] +
    merged_data['B15002_018E_diff'] +
    merged_data['B15002_032E_diff'] +
    merged_data['B15002_033E_diff'] +
    merged_data['B15002_034E_diff'] +
    merged_data['B15002_035E_diff']
)

# replace the column names with more human-readable names
merged_data.rename(columns={
    'B25064_001E_2010_on_2020': 'median_gross_rent_2010_on_2020',
    'B19013_001E_2010_on_2020': 'median_household_income_2010_on_2020',
    'B19013_001E_2020': 'median_household_income_2020',
    'B19013_001E_diff': 'median_household_income_diff',
    'B17021_001E_diff': 'poverty_population_diff',
    'B25064_001E_diff': 'median_gross_rent_diff',
    'B25077_001E_diff': 'median_home_value_diff',
    'B25003_002E_diff': 'owner_occupied_housing_units_diff',
    'B25003_003E_diff': 'renter_occupied_housing_units_diff',

}, inplace=True)

# select only the relevant columns for the final dataset
merged_data = merged_data[[
    'GEOID',
    'poverty_population_diff',
    'median_household_income_diff',
    'median_gross_rent_2010_on_2020',
    'median_gross_rent_diff',
    'median_home_value_diff',
    'bachelors_or_higher_diff',
    'median_household_income_2010_on_2020',
    'median_household_income_2020'
]]

# Save the merged dataset to a new CSV file
merged_data.to_csv("merged_census_data.csv", index=False)

Assigning Census Tracts ID values to crime statistics. May not be necessary - might deprecate in future version.

In [ ]:

# US Census Tracts for Illinois (state FIPS code 17) from the 2020 TIGER/Line shapefiles
tracts = gpd.read_file("unprocessed_data/tl_2020_17_tract/tl_2020_17_tract.shp")
city_boundary = gpd.read_file('unprocessed_data\city-shapefile\geo_export_41a1e3c1-d168-4dc6-9f14-0e861c9ee606.shp')

def coordinate_to_census_tract(lat, lon, city_boundary):
    """Given a latitude and longitude, return the corresponding Census tract GEOID."""
    if (math.isnan(lat)) or (math.isnan(lon)):
        return pd.NA
    point = Point(lon, lat)
    target_tract = tracts[tracts.contains(point)]
    if target_tract.empty:
        return pd.NA
    # Check if the point is within the city boundary. If not, return NA.
    elif not target_tract.iloc[0].geometry.within(city_boundary.geometry.iloc[0]):
        return pd.NA
    else:
        return target_tract['TRACTCE'].iloc[0]

def add_tract_geoid(df, lat_field, lon_field):
    """Add a 'GEOID' column to the DataFrame by converting latitude and longitude to Census tract GEOIDs."""
    df = df.copy()
    df['GEOID'] = df.apply(lambda row: coordinate_to_census_tract(row[lat_field], row[lon_field], city_boundary), axis=1)
    return df

for chunk in pd.read_csv('Crime_Data_2006_2010.csv', chunksize=10000):
    chunk_with_geoid = add_tract_geoid(chunk, 'latitude', 'longitude')
    chunk_with_geoid = chunk_with_geoid[~chunk_with_geoid['GEOID'].isna()]
    chunk_with_geoid.to_csv('Crime_Data_2006_2010_edit.csv', index=False, mode='a', header=True if chunk_with_geoid.index[0] == 0 else False)

for chunk in pd.read_csv('Crime_Data_2016_2020.csv', chunksize=10000):
    chunk_with_geoid = add_tract_geoid(chunk, 'latitude', 'longitude')
    chunk_with_geoid = chunk_with_geoid[~chunk_with_geoid['GEOID'].isna()]
    chunk_with_geoid.to_csv('Crime_Data_2016_2020_edit.csv', index=False, mode='a', header=True if chunk_with_geoid.index[0] == 0 else False)

Filtering Census Data to exclude information not located within your city boundaries. Requires a city boundary shapefile.

In [ ]:
census_data = pd.read_csv('merged_census_data.csv')

# filter out rows in which the GEOID is not in the city shapefile
mask = census_data["GEOID"].apply(
    lambda x: census_tract_overlaps_city(x, city_boundary)
)

census_data = census_data[mask]
census_data = census_data[(census_data['median_household_income_2010_on_2020'] <= median_income) & (census_data['median_gross_rent_2010_on_2020'] <= median_rent)]

census_data.to_csv('merged_census_data_filtered.csv', index=False)

Calculating the overall historical gentrification scores using a z-score weighted sum. This score is used as a sanity check to determine if the factors measured represent areas that have undergone gentrification in the city.

In [ ]:
census_data = pd.read_csv('merged_census_data_filtered.csv')

census_data['gentrification_score'] = 0  # Initialize the gentrification score column
census_data_zscore = census_data.copy()
for column in census_data.columns:
    if column == 'GEOID' or column == 'gentrification_score' or column == 'median_household_income_2010_on_2020' or column == 'median_household_income_2020' or column == 'median_gross_rent_2010_on_2020':  # Skip the GEOID and median income columns
        continue
    elif column == 'poverty_population_diff':  # Invert the poverty population difference so that a decrease in poverty contributes positively to gentrification
        census_data_zscore[column] = (census_data[column] - census_data[column].mean()) / census_data[column].std()
        census_data['gentrification_score'] -= census_data_zscore[column]
    else:  # Don't normalize the GEOID column
        census_data_zscore[column] = (census_data[column] - census_data[column].mean()) / census_data[column].std()

        # Calculate gentrification score as a weighted sum of z-scores
        census_data['gentrification_score'] += census_data_zscore[column]

census_data.to_csv('merged_census_data_gentrification_scored.csv', index=False)




In [ ]:
jobs_by_census_tract_2010[['latitude', 'longitude']] = jobs_by_census_tract_2010['w_geocode'].apply(
    lambda geoid: pd.Series(census_tract_to_coordinates(geoid)))

jobs_by_census_tract_2010.to_csv('jobs_by_census_tract_2010_with_coordinates.csv', index=False)

In [ ]:
parcel_data = pd.read_csv('data/merged_parcel_data_2010-2020.csv')
parcel_data['GEOID'] = parcel_data.apply(lambda row: coordinates_to_census_tract(row['latitude_x'], row['longitude_x']), axis=1)
parcel_data = parcel_data[~parcel_data['GEOID'].isna()] 
parcel_data.to_csv('merged_parcel_data_2010-2020_with_geoid.csv', index=False)

In [ ]:
# truncate our crime dataset coordinates down to 3 decimal places, then agglomerate how many crimes occurred per year in each coordinate pair
parcel_data = pd.read_csv('data/crime/Crime_Data_2006_2010.csv')

parcel_data['latitude'] = parcel_data['latitude'].round(3)
parcel_data['longitude'] = parcel_data['longitude'].round(3)
crime_counts = parcel_data.groupby(['latitude', 'longitude', 'year']).size().reset_index(name='crime_count')
crime_counts.to_csv('data/crime/crime_counts_by_coordinates_and_year_2006_2010.csv', index=False)

Now that we have our prepared dataset with gentrification scores, we can start building the predictive model.
Our goal is to assign particular weights to each of the following, to determine their overall role in the gentrification process:
- distance from jobs
- distance from park facilities
- distance from nearest rail station
- building permit density + trends
- crime density + trends

Our approach will be as follows - for each of the census tracts in the dataset:
- analyze the factors above
- adjust the weights accordingly

This process requires a lot of pre-processing beforehand to obtain the lat/long coordinates from the tract IDs, but overall doesn't take that long to complete (~100min for Chicago)

In [ ]:
training_data = pd.read_csv('merged_census_data_gentrification_scored.csv')
import matplotlib.pyplot as plt

# remove tracts with missing gentrification scores
training_data = training_data[~training_data['gentrification_score'].isna()]

# mark the top 10% of census tracts by gentrification score as gentrifying (label = 1). The rest are non-gentrifying (label = 0).
training_data['gentrifying'] = (training_data['gentrification_score'] >= training_data['gentrification_score'].quantile(0.9)).astype(int)

# get a 100000 random sample of the crime data with geocoded census tracts
#Crime_Data_2006_2010 = pd.read_csv('data/Crime_Data_2006_2010.csv').sample(n=100000, random_state=42)
#Building_Permits_2006_2010 = pd.read_csv('data/Building_Permits_2006_2010.csv')
jobs_by_census_tract_2010 = pd.read_csv('data/jobs_by_census_tract_2010_with_coordinates.csv')

Crime_Data_2006_2010_date_adjusted = Crime_Data_2006_2010.copy()
Building_Permits_2006_2010_date_adjusted = Building_Permits_2006_2010.copy()
Crime_Data_2006_2010_date_adjusted['timestamp'] = Crime_Data_2006_2010_date_adjusted['timestamp'].apply(lambda x: x[0:2] + x[5:10])
Crime_Data_2006_2010_date_adjusted['timestamp'] = pd.to_datetime(Crime_Data_2006_2010_date_adjusted['timestamp'], format='%m/%Y').astype('int64') // 10**9
Building_Permits_2006_2010_date_adjusted['date'] = Building_Permits_2006_2010_date_adjusted['date'].apply(lambda x: x[6:10])
parcel_data = pd.read_csv('merged_parcel_data_2010-2020_with_geoid.csv')

# output CSV file setup with columns
with open('census_tract_features.csv', 'a') as f:
    f.write('GEOID,latitude,longitude,job_gravity,rail_gravity,park_gravity,crime_gravity,crime_trend,building_permit_gravity,building_permit_trend,gentrifying\n')

for tract in training_data['GEOID']:
    # lat, lon = census_tract_to_coordinates(tract)
    # if pd.isna(lat) or pd.isna(lon):
    #     print(f"Skipping tract {tract} due to missing coordinates.")
    #     continue
    # get every parcel in the tract
    tract_parcels = parcel_data[parcel_data['GEOID'] == tract]
    parcels_gravities = pd.DataFrame()
    for parcel in tract_parcels.itertuples():

        lat = parcel.latitude_x
        lon = parcel.longitude_x
    
        # determine gravity from jobs, weighted by distance
        job_gravity = 0
        jobs_by_census_tract_2010['distance'] = jobs_by_census_tract_2010.apply(
                lambda x: math.sqrt((lat - x['latitude'])**2 + (lon - x['longitude'])**2), axis=1)
        job_gravity = calculate_gravity_score(jobs_by_census_tract_2010, distance=math.inf)
        

        # determine gravity from rail stations, weighted by distance
        rail_gravity = 0
        rail_stations['distance'] = rail_stations.apply(
                lambda x: math.sqrt((lat - x['Y'])**2 + (lon - x['X'])**2), axis=1)
        rail_gravity = calculate_gravity_score(rail_stations)

        # determine gravity from park facilities, weighted by distance
        park_gravity = 0
        park_facilities['distance'] = park_facilities.apply(
                lambda x: math.sqrt((lat - x['Y_COORD'])**2 + (lon - x['X_COORD'])**2), axis=1)
        park_gravity = calculate_gravity_score(park_facilities)

        # find the number of crimes committed over the last 2 years of t_0, weighted by distance
        # crime_gravity = 0
        Crime_Data_2006_2010_date_adjusted['distance'] = Crime_Data_2006_2010_date_adjusted.apply(
            lambda crime: math.sqrt((lat - crime['latitude'])**2 + (lon - crime['longitude'])**2), axis=1)
        # crime_gravity = calculate_gravity_score(Crime_Data_2006_2010_date_adjusted, year_var='year', min_year=2008)

        crime_counts = Crime_Data_2006_2010_date_adjusted[Crime_Data_2006_2010_date_adjusted['distance'] < distance_threshold].groupby('year').size()
        # use linear regression to find the slope of crime over time
        crime_model = LinearRegression()
        crime_model.fit(crime_counts.index.values.reshape(-1, 1), crime_counts.values)

        # multiply the crime gravity by the slope of crime over time to get a combined measure of crime intensity and trend
        # plt.scatter(crime_counts.index.values, crime_counts.values, color='blue', label='Crime Counts')
        # plt.plot(crime_counts.index.values.reshape(-1,1), crime_model.predict(crime_counts.index.values.reshape(-1,1)), color='red', label='Regression Line')
        # plt.show()

        # determine gravity from building permits, weighted by distance
        building_permit_gravity = 0
        Building_Permits_2006_2010_date_adjusted['distance'] = Building_Permits_2006_2010_date_adjusted.apply(
            lambda permit: math.sqrt((lat - permit['latitude'])**2 + (lon - permit['longitude'])**2), axis=1)
        building_permit_gravity = calculate_gravity_score(Building_Permits_2006_2010_date_adjusted, year_var='date', min_year=2008)

        permit_counts = Building_Permits_2006_2010_date_adjusted[Building_Permits_2006_2010_date_adjusted['distance'] < distance_threshold].groupby('date').size()
        # use linear regression to find the slope of crime over time
        permit_model = LinearRegression()
        permit_model.fit(permit_counts.index.values.reshape(-1, 1), permit_counts.values)

        # plt.scatter(permit_counts.index.values, permit_counts.values, color='green', label='Permit Counts')
        # plt.plot(permit_counts.index.values.reshape(-1,1), permit_model.predict(permit_counts.index.values.reshape(-1,1)), color='orange', label='Regression Line')
        # plt.show()

        parcels_gravities = pd.concat([parcels_gravities, pd.DataFrame({
            'job_gravity': job_gravity, 'rail_gravity': rail_gravity, 
            'park_gravity': park_gravity,
            'crime_trend': crime_model.coef_[0], 'building_permit_gravity': building_permit_gravity,
              'building_permit_trend': permit_model.coef_[0]}, index=[0])])

    mean_gravities = parcels_gravities.mean()
    median_gravities = parcels_gravities.median()
    percentile90_gravities = parcels_gravities.quantile(0.9)
    
    with open('census_tract_features_mean_gravities.csv', 'a') as f:
        f.write(f'{tract},{lat},{lon},{mean_gravities["job_gravity"]},{mean_gravities["rail_gravity"]},{mean_gravities["park_gravity"]},{mean_gravities["crime_gravity"]},{mean_gravities["crime_trend"]},{mean_gravities["building_permit_gravity"]},{mean_gravities["building_permit_trend"]},{training_data[training_data["GEOID"] == tract]["gentrifying"].values[0]}\n'.format())  


    with open('census_tract_features_median_gravities.csv', 'a') as f:
        f.write(f'{tract},{lat},{lon},{median_gravities["job_gravity"]},{median_gravities["rail_gravity"]},{median_gravities["park_gravity"]},{median_gravities["crime_gravity"]},{median_gravities["crime_trend"]},{median_gravities["building_permit_gravity"]},{median_gravities["building_permit_trend"]},{training_data[training_data["GEOID"] == tract]["gentrifying"].values[0]}\n'.format())  

    with open('census_tract_features_p90_gravities.csv', 'a') as f:
        f.write(f'{tract},{lat},{lon},{percentile90_gravities["job_gravity"]},{percentile90_gravities["rail_gravity"]},{percentile90_gravities["park_gravity"]},{percentile90_gravities["crime_gravity"]},{percentile90_gravities["crime_trend"]},{percentile90_gravities["building_permit_gravity"]},{percentile90_gravities["building_permit_trend"]},{training_data[training_data["GEOID"] == tract]["gentrifying"].values[0]}\n'.format())  





C:\Users\gilre\AppData\Local\Temp\ipykernel_17004\159587944.py:20: DtypeWarning: Columns (0: class_x, 1: class_y) have mixed types. Specify dtype option on import or set low_memory=False.
  parcel_data = pd.read_csv('merged_parcel_data_2010-2020_with_geoid.csv')


In [ ]:
census_tract_features = pd.read_csv('data/census_tract_features.csv')
# standardize the features
for column in census_tract_features.columns:
    if column in ['GEOID', 'latitude', 'longitude', 'gentrifying']:
        continue
    elif column in ['crime_trend', 'building_permit_trend']:
        # flip the signs in trend features for interpretability, so that an increasing crime trend contributes negatively to gentrification and an increasing building permit trend contributes positively to gentrification
        census_tract_features[column] = -census_tract_features[column]
    census_tract_features[column] = (census_tract_features[column] - census_tract_features[column].mean()) / census_tract_features[column].std()

census_tract_features.to_csv('data/census_tract_features_standardized.csv', index=False)

In [ ]:
# Readjust gentrified labels if necessary based on gentrification scores from earlier
# census_tract_features = pd.read_csv('data/census_tract_features_standardized.csv')
# training_data = pd.read_csv('data/merged_census_data_gentrification_scored.csv')
# census_tract_features['gentrifying'] = census_tract_features['GEOID'].apply(lambda x: not math.isnan(training_data[training_data['GEOID'] == x]['gentrification_score'].values[0]))
# census_tract_features.to_csv('data/census_tract_features_standardized.csv', index=False)

Now that we have our gravity observations, it's time to train our weights!

In [ ]:
import pymc as pm
import arviz as az

# Bayesian inference approach to updating the weights based on the observed data.
observations = pd.read_csv('data/census_tract_features_standardized.csv')
    
feature_cols = [
    "job_gravity",
    "rail_gravity",
    "park_gravity",
    "crime_gravity",
    "crime_trend", # signifies relative value to other tracts (higher value means better on crime trend than the mean tract)
    "building_permit_gravity",
    "building_permit_trend", # signifies relative value to other tracts (higher value means better on building permit trend than the mean tract)
]

X = observations[feature_cols].copy()
Y = observations["gentrifying"].astype(int).values

X_mat = X.values
n_features = X_mat.shape[1]

with pm.Model() as model:
    # Priors
    intercept = pm.Normal("intercept", mu=0, sigma=1)
    weights = pm.Normal("weights", mu=0, sigma=1, shape=n_features)

    # Linear predictor
    eta = intercept + pm.math.dot(X_mat, weights)

    # Logistic link
    p = pm.Deterministic("p", pm.math.sigmoid(eta))

    # Likelihood
    y_obs = pm.Bernoulli("y_obs", p=p, observed=Y)

    # Posterior sampling
    trace = pm.sample(
        draws=2000,
        tune=2000,
        target_accept=0.9,
        chains=4,
        random_seed=42
    )
    
print(az.summary(trace, var_names=["intercept", "weights"]))


With the weights set up, begin training with BoTorch!